# 03 Gold - Business questions

The eight demo questions, answered in SQL on the Gold star schema. These are the **reference answers**: the
Kenseme Data Agent and the Fabric Data Agent must return the same figures.

Every query follows the same pattern: start from a **fact**, join the **dimensions** you slice by, aggregate.
Revenue is always `LineAmount` (lines) or `SubTotal` (orders): before tax, after discounts.

The last section compares the layers: the same question on Silver vs Gold, and totals from Bronze to Gold.
Read only. Run cell by cell, or **Run all**.

## Q1. Revenue by product category group and year
Online sales run 2019-2021, reseller sales are 2022, so 2022 shows the reseller mix.
Expected: Bikes grows 3.86M → 6.37M → 10.17M; Accessories and Clothing appear in 2021.

In [ ]:
%%sql
SELECT d.CalendarYear, c.ProductCategoryGroup, ROUND(SUM(l.LineAmount), 2) AS Revenue
FROM AdventureWorksGold.dbo.FactSalesOrderLine l
JOIN AdventureWorksGold.dbo.FactSalesOrder o      ON o.SalesOrderKey = l.SalesOrderKey
JOIN AdventureWorksGold.dbo.DimDate d             ON d.DateKey = o.OrderDateKey
JOIN AdventureWorksGold.dbo.DimProduct p          ON p.ProductKey = l.ProductKey
JOIN AdventureWorksGold.dbo.DimProductCategory c  ON c.ProductCategoryKey = p.ProductCategoryKey
GROUP BY d.CalendarYear, c.ProductCategoryGroup
ORDER BY d.CalendarYear, Revenue DESC

## Q2. Top 5 bike models by revenue
`ProductModel` rolls all sizes and colors of a model together (parsed in Silver).
Expected: Road-150 5,549,896.77 (1,551 units), Mountain-200, Road-250, Touring-1000, Mountain-100.

In [ ]:
%%sql
SELECT p.ProductModel, ROUND(SUM(l.LineAmount), 2) AS Revenue, SUM(l.Quantity) AS Units
FROM AdventureWorksGold.dbo.FactSalesOrderLine l
JOIN AdventureWorksGold.dbo.DimProduct p          ON p.ProductKey = l.ProductKey
JOIN AdventureWorksGold.dbo.DimProductCategory c  ON c.ProductCategoryKey = p.ProductCategoryKey
WHERE c.ProductCategoryGroup = 'Bikes'
GROUP BY p.ProductModel ORDER BY Revenue DESC LIMIT 5

## Q3. Orders, revenue and average order value by channel
Expected: Online 16,459 orders, 20,928,022.43, AOV 1,271.52; Reseller 32 orders, 708,690.07, AOV 22,146.56.

In [ ]:
%%sql
SELECT ch.SalesChannelName, COUNT(*) AS Orders, ROUND(SUM(o.SubTotal), 2) AS Revenue,
       ROUND(SUM(o.SubTotal) / COUNT(*), 2) AS AvgOrderValue
FROM AdventureWorksGold.dbo.FactSalesOrder o
JOIN AdventureWorksGold.dbo.DimSalesChannel ch ON ch.SalesChannelKey = o.SalesChannelKey
GROUP BY ch.SalesChannelName

## Q4. Top 3 online customers
Filter on `CustomerType = 'Individual'`: across both channels the top 3 are reseller accounts.
Expected: Larry Vazquez 10,899.62, Kaitlyn Henderson 10,896.22, Nichole Nara 10,876.32 (4 orders each).

In [ ]:
%%sql
SELECT c.CustomerName, COUNT(*) AS Orders, ROUND(SUM(o.SubTotal), 2) AS Revenue
FROM AdventureWorksGold.dbo.FactSalesOrder o
JOIN AdventureWorksGold.dbo.DimCustomer c ON c.CustomerKey = o.CustomerKey
WHERE c.CustomerType = 'Individual'
GROUP BY c.CustomerName ORDER BY Revenue DESC LIMIT 3

## Q5. Online orders per month, January to August 2021
The story: orders jump from ~300 to ~1,600 a month in June-July, while revenue barely moves. Accessories launched
mid-2021: many small orders.

In [ ]:
%%sql
SELECT d.YearMonth, COUNT(*) AS Orders, ROUND(SUM(o.SubTotal), 2) AS Revenue,
       ROUND(SUM(o.SubTotal) / COUNT(*), 2) AS AvgOrderValue
FROM AdventureWorksGold.dbo.FactSalesOrder o
JOIN AdventureWorksGold.dbo.DimDate d ON d.DateKey = o.OrderDateKey
WHERE o.SalesChannelKey = 1 AND d.YearMonth BETWEEN '2021-01' AND '2021-08'
GROUP BY d.YearMonth ORDER BY d.YearMonth

## Q6. Lines sold below list price, and total discount, by channel
Expected: Online 2,449 of 32,718 lines, 528,750.88; Reseller all 542 lines, 509,928.20.

In [ ]:
%%sql
SELECT ch.SalesChannelName, SUM(CASE WHEN l.DiscountAmount > 0 THEN 1 ELSE 0 END) AS DiscountedLines,
       COUNT(*) AS Lines, ROUND(SUM(l.DiscountAmount), 2) AS TotalDiscount
FROM AdventureWorksGold.dbo.FactSalesOrderLine l
JOIN AdventureWorksGold.dbo.FactSalesOrder o   ON o.SalesOrderKey = l.SalesOrderKey
JOIN AdventureWorksGold.dbo.DimSalesChannel ch ON ch.SalesChannelKey = o.SalesChannelKey
GROUP BY ch.SalesChannelName

## Q7. Repeat online customers
Expected: 3,310 of 12,427 online customers (26.6%) ordered more than once.

In [ ]:
%%sql
SELECT SUM(CASE WHEN n > 1 THEN 1 ELSE 0 END) AS RepeatCustomers, COUNT(*) AS Customers,
       ROUND(100.0 * SUM(CASE WHEN n > 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS RepeatPct
FROM (SELECT CustomerKey, COUNT(*) AS n FROM AdventureWorksGold.dbo.FactSalesOrder
      WHERE SalesChannelKey = 1 GROUP BY CustomerKey)

## Q8. Top 5 categories by units sold in 2021
Expected: Tires and Tubes 8,750, Bottles and Cages 3,945, Helmets 3,152, Road Bikes 2,956, Mountain Bikes 2,279.

In [ ]:
%%sql
SELECT c.ProductCategoryName, SUM(l.Quantity) AS Units, ROUND(SUM(l.LineAmount), 2) AS Revenue
FROM AdventureWorksGold.dbo.FactSalesOrderLine l
JOIN AdventureWorksGold.dbo.FactSalesOrder o      ON o.SalesOrderKey = l.SalesOrderKey
JOIN AdventureWorksGold.dbo.DimDate d             ON d.DateKey = o.OrderDateKey
JOIN AdventureWorksGold.dbo.DimProduct p          ON p.ProductKey = l.ProductKey
JOIN AdventureWorksGold.dbo.DimProductCategory c  ON c.ProductCategoryKey = p.ProductCategoryKey
WHERE d.CalendarYear = 2021
GROUP BY c.ProductCategoryName ORDER BY Units DESC LIMIT 5

## Why Gold? The same question on Silver
Q3 (revenue by channel) on Silver needs a different query per source, its own revenue formula for each, and a
UNION. On Gold it is one join. Every consumer (reports, Kenseme, agents) gets the same answer from Gold because
the rules live in one place.

In [ ]:
%%sql
SELECT 'Online' AS channel, COUNT(DISTINCT SalesOrderNumber) AS Orders, ROUND(SUM(UnitPrice * Quantity), 2) AS Revenue
FROM AdventureWorksSilver.dbo.online_sales_line
UNION ALL
SELECT 'Reseller', COUNT(DISTINCT SalesOrderNumber), ROUND(SUM(LineAmount), 2)
FROM AdventureWorksSilver.dbo.reseller_order_line

## From Bronze to Gold: one row per layer
Online lines are counted twice in Bronze (four files), once in Silver and Gold. Orders and revenue match from
Silver on.

In [ ]:
from pyspark.sql import functions as F

def lakehouse_path(name):
    """OneLake path of a lakehouse in this workspace, found by name."""
    return notebookutils.lakehouse.get(name)["properties"]["abfsPath"]

BRONZE = lakehouse_path("FabricWorkshopDemo")  # read only
cols = ["SalesOrderNumber", "SalesOrderLineNumber", "OrderDate", "CustomerName", "EmailAddress", "Item", "Quantity", "UnitPrice", "TaxAmount"]
csv = lambda p, h: spark.read.option("header", h).option("quote", '"').option("escape", '"').csv(f"{BRONZE}/Files/{p}")
bronze_online = csv("data/sales.csv", True)
for y in ("2019", "2020", "2021"):
    bronze_online = bronze_online.unionByName(csv(f"bronze/{y}.csv", False).toDF(*cols))
bronze_online.createOrReplaceTempView("bronze_online_lines")

In [ ]:
%%sql
SELECT 'Bronze' AS layer, 'online lines, 4 files' AS source, COUNT(*) AS lines, COUNT(DISTINCT SalesOrderNumber) AS orders,
       ROUND(SUM(CAST(UnitPrice AS DECIMAL(19,4)) * CAST(Quantity AS INT)), 2) AS revenue
FROM bronze_online_lines
UNION ALL
SELECT 'Bronze', 'reseller orders table', COUNT(*), COUNT(DISTINCT SalesOrderID), ROUND(SUM(LineItemTotal), 2)
FROM FabricWorkshopDemo.dbo.orders
UNION ALL
SELECT 'Silver', 'online_sales_line', COUNT(*), COUNT(DISTINCT SalesOrderNumber), ROUND(SUM(UnitPrice * Quantity), 2)
FROM AdventureWorksSilver.dbo.online_sales_line
UNION ALL
SELECT 'Silver', 'reseller_order_line', COUNT(*), COUNT(DISTINCT SalesOrderNumber), ROUND(SUM(LineAmount), 2)
FROM AdventureWorksSilver.dbo.reseller_order_line
UNION ALL
SELECT 'Gold', 'FactSalesOrderLine (both channels)', COUNT(*), COUNT(DISTINCT SalesOrderKey), ROUND(SUM(LineAmount), 2)
FROM AdventureWorksGold.dbo.FactSalesOrderLine